ADTOF automático para o Compasso. Execute as quatro células: instalação, envio do áudio, transcrição, alinhamento/exportação. Não é necessário informar BPM, fator ou início do compasso. O rastreador estima pulsação e compassos em 4/4, com mudanças de andamento, e leva o alinhamento no MIDI. O app aplica esse alinhamento ao importar. Use a mesma gravação e o mesmo início do vídeo. A análise pode errar, especialmente em ritmos ambíguos; não há garantia de correção musical. Golpes anteriores à primeira batida detectada são preservados, extrapolando a grade. Salve uma cópia no Drive e baixe o MIDI; o armazenamento temporário do Colab pode ser apagado.
Esta versão preserva os compassos da introdução a partir do início da gravação, mesmo sem golpes de bateria. O alinhamento residual é próximo do início do áudio; notas antecipadas são preservadas. Abra novamente o notebook do GitHub para obter esta atualização: cópias antigas no Drive não são atualizadas automaticamente.

In [ ]:
#@title Preparar ambiente
import sys, subprocess, os
from pathlib import Path

BASE = Path('/content/adtof-testes')
BASE.mkdir(parents=True, exist_ok=True)
PYTHON = BASE / 'ambiente/bin/python'

def executar(comando, **opcoes):
    resultado = subprocess.run(
        [str(x) for x in comando], stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, **opcoes
    )
    print(resultado.stdout)
    resultado.check_returncode()

print('Preparando o ADTOF. A primeira instalação pode levar alguns minutos…')
executar([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = [sys.executable, '-m', 'uv']
if not PYTHON.exists():
    executar(UV + ['venv', '--python', '3.10', '--seed', BASE / 'ambiente'])

versoes = BASE / 'versoes.txt'
versoes.write_text('''numpy==1.23.5
scipy==1.10.1
Cython==0.29.37
setuptools==69.5.1
wheel==0.45.1
tensorflow==2.15.1
keras==2.15.0
librosa==0.10.2.post1
numba==0.60.0
matplotlib==3.8.4
pandas==2.2.3
scikit-learn==1.3.2
''')
instalar = UV + ['pip', 'install', '--python', PYTHON, '-c', versoes]
executar(instalar + [
    'numpy', 'scipy', 'Cython', 'setuptools', 'wheel', 'tensorflow',
    'librosa', 'numba', 'matplotlib', 'pandas', 'scikit-learn'
])
executar(instalar + ['--no-build-isolation',
    'madmom==0.16.1', 'mir_eval', 'jellyfish', 'pyunpack',
    'ffmpeg-python', 'pretty_midi', 'argparse', 'beautifulsoup4', 'mido'
])
executar(instalar + ['--no-deps', '--no-build-isolation',
    'git+https://github.com/MZehren/ADTOF@b3968fb332f69b65ee07c089fc62f436503755db',
    'git+https://github.com/MZehren/tapcorrect@4f2d21e73fb0137119a4136513c42936b322fc0b#subdirectory=python'
])
print('Instalação concluída. Pode executar a célula 2; não precisa reiniciar.')


In [ ]:
#@title Enviar a música
from google.colab import files
import uuid

print('Escolha somente um MP3 ou WAV da mesma gravação usada no vídeo.')
enviados = files.upload()
if len(enviados) != 1:
    raise ValueError('Envie um único arquivo por teste.')

nome, dados = next(iter(enviados.items()))
if Path(nome).suffix.lower() not in ('.mp3', '.wav'):
    raise ValueError('Use um arquivo MP3 ou WAV.')
PASTA = BASE / ('teste-' + uuid.uuid4().hex[:8])
PASTA.mkdir()
AUDIO = PASTA / ('audio' + Path(nome).suffix.lower())
AUDIO.write_bytes(dados)
print('Áudio recebido:', nome)


In [ ]:
#@title Transcrever a bateria
# Execute esta célula depois de enviar o áudio.
# Cada execução tem uma pasta nova, evitando resultados antigos.
SAIDA = PASTA / ('resultado-' + uuid.uuid4().hex[:8])

codigo = r'''
import os, sys, collections, collections.abc
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_NUM_INTRAOP_THREADS'] = '2'
os.environ['TF_NUM_INTEROP_THREADS'] = '2'
collections.MutableSequence = collections.abc.MutableSequence

from adtof.model.model import Model
print('Carregando o modelo treinado…', flush=True)
model, hparams = Model.modelFactory(
    modelName='Frame_RNN', scenario='adtofAll', fold=0
)
if not model.weightLoadedFlag:
    raise RuntimeError('Os pesos treinados não foram encontrados.')
print('Transcrevendo a bateria…', flush=True)
model.predictFolder(sys.argv[1], sys.argv[2], writeMidi=False, **hparams)
arquivos = list(Path(sys.argv[2]).glob('*.txt'))
if len(arquivos) != 1 or not arquivos[0].read_text().strip():
    raise RuntimeError('Não foi produzida uma transcrição com notas. Confira o áudio e as mensagens acima.')
print('Transcrição concluída.')
'''
executar([PYTHON, '-u', '-c', codigo, AUDIO, SAIDA])
TXT = next(SAIDA.glob('*.txt'))
print('Resultado:', TXT.name)


In [ ]:
#@title Gerar MIDI com alinhamento automático
# Detecta a pulsação, alinha os compassos e exporta o MIDI automaticamente.
# Use o áudio da mesma gravação, com o mesmo início do vídeo no app.
import os
if 'TXT' not in globals() or TXT.parent != SAIDA or SAIDA.parent != AUDIO.parent:
    raise RuntimeError('Execute a transcrição da célula 3 para este áudio antes de exportar.')
codigo_automatico = '"""Align ADTOF attacks to detected 4/4 beats without moving audio timestamps.\n\nThe MIDI marker carries the time of tick zero in the source recording. Compasso\napplies it on import. Tempo events preserve detected beat times, including tempo\nchanges; attacks are not quantized here. Compasso\'s display uses sixteenths.\n"""\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom mido import MidiFile, MidiTrack, Message, MetaMessage\n\nPPQ = 960\nMARKER = \'COMPASSO_SYNC_V1:\'\n\n\ndef extrapolate(values, xp, fp):\n    values = np.asarray(values, dtype=float)\n    out = np.interp(values, xp, fp)\n    before, after = values < xp[0], values > xp[-1]\n    out[before] = fp[0] + (values[before]-xp[0]) * (fp[1]-fp[0])/(xp[1]-xp[0])\n    out[after] = fp[-1] + (values[after]-xp[-1]) * (fp[-1]-fp[-2])/(xp[-1]-xp[-2])\n    return out\n\n\ndef export_aligned(notes, beats, output):\n    notes, beats = np.asarray(notes, dtype=float), np.asarray(beats, dtype=float)\n    if notes.ndim != 2 or notes.shape[1] != 2 or not 1 <= len(notes) <= 20000:\n        raise ValueError(\'A transcrição deve conter de 1 a 20 mil golpes.\')\n    if not np.isfinite(notes).all() or np.any(notes[:, 0] < 0):\n        raise ValueError(\'Transcrição com tempos inválidos.\')\n    if not np.isin(notes[:, 1], [35, 38, 42, 47, 49]).all():\n        raise ValueError(\'Este exportador espera as cinco classes do ADTOF.\')\n    if beats.ndim != 2 or beats.shape[1] != 2 or len(beats) < 8 or not np.isfinite(beats).all():\n        raise ValueError(\'Não foi possível identificar pelo menos dois compassos.\')\n    if np.any(np.diff(beats[:, 0]) <= 0) or not np.isin(beats[:, 1], [1, 2, 3, 4]).all():\n        raise ValueError(\'O mapa de compassos detectado é inválido.\')\n    if not np.all((np.diff(beats[:, 1]) % 4) == 1):\n        raise ValueError(\'A detecção deixou lacunas nos tempos. O alinhamento não é confiável.\')\n\n    # Coordinates are quarter notes, anchored to the tracker\'s downbeats.\n    coordinates = np.arange(len(beats)) + beats[0, 1] - 1\n    notes = notes[np.argsort(notes[:, 0], kind=\'stable\')]\n    musical_positions = extrapolate(notes[:, 0], beats[:, 0], coordinates)\n    # Number bars from the recording\'s beginning, not from the first drum hit.\n    # Otherwise a 12-bar intro disappears and becomes a large playback offset.\n    # Retain pickup notes too, even if that requires a bar starting before zero.\n    epsilon = 1e-9\n    position_at_zero = float(extrapolate([0], beats[:, 0], coordinates)[0])\n    first_at_audio = math.ceil((position_at_zero-epsilon)/4) * 4\n    first_at_notes = math.floor((float(np.min(musical_positions))+epsilon)/4) * 4\n    first = min(first_at_audio, first_at_notes)\n    last = math.floor((float(np.max(musical_positions))+epsilon)/4) * 4 + 4\n    if (last-first)//4 > 500:\n        raise ValueError(\'A música excede o limite atual de 500 compassos do app.\')\n    boundaries = extrapolate(np.arange(first, last+1), coordinates, beats[:, 0])\n    offset = float(boundaries[0])\n    if abs(offset) > 1800:\n        raise ValueError(\'Alinhamento fora do limite de 30 minutos do app.\')\n\n    midi = MidiFile(ticks_per_beat=PPQ)\n    metadata, drums = MidiTrack(), MidiTrack()\n    midi.tracks.extend([metadata, drums])\n    metadata.append(MetaMessage(\'track_name\', name=\'Compasso · alinhamento automático\', time=0))\n    metadata.append(MetaMessage(\'marker\', text=MARKER + f\'{offset:.6f}\', time=0))\n    metadata.append(MetaMessage(\'time_signature\', numerator=4, denominator=4, time=0))\n    previous_tick, previous_tempo = 0, None\n    for i, seconds in enumerate(np.diff(boundaries)):\n        tempo = round(float(seconds)*1_000_000)\n        if not 1 <= tempo <= 16777215:\n            raise ValueError(\'Andamento detectado fora do intervalo MIDI.\')\n        if tempo != previous_tempo:\n            metadata.append(MetaMessage(\'set_tempo\', tempo=tempo, time=i*PPQ-previous_tick))\n            previous_tick, previous_tempo = i*PPQ, tempo\n    metadata.append(MetaMessage(\'end_of_track\', time=0))\n\n    events = []\n    durations = np.full(len(notes), 0.03)\n    next_by_pitch = {}\n    for i in range(len(notes)-1, -1, -1):\n        seconds, pitch = notes[i]\n        if pitch in next_by_pitch:\n            durations[i] = min(0.03, max(0.001, (next_by_pitch[pitch]-seconds)/2))\n        next_by_pitch[pitch] = seconds\n    for i, ((seconds, pitch), position) in enumerate(zip(notes, musical_positions)):\n        start = round((position-first)*PPQ)\n        duration = durations[i]\n        end_position = extrapolate([seconds+max(duration, 0.001)], beats[:, 0], coordinates)[0]\n        end = max(start+1, round((end_position-first)*PPQ))\n        events.extend([(start, 1, int(pitch)), (end, 0, int(pitch))])\n    previous_tick = 0\n    for tick, on, pitch in sorted(events):\n        drums.append(Message(\'note_on\' if on else \'note_off\', channel=9, note=pitch,\n                             velocity=90 if on else 0, time=tick-previous_tick))\n        previous_tick = tick\n    drums.append(MetaMessage(\'end_of_track\', time=0))\n    midi.save(output)\n    return {\'offset\': offset, \'median_bpm\': float(60/np.median(np.diff(beats[:, 0]))),\n            \'notes\': len(notes), \'bars\': (last-first)//4,\n            \'leading_bars\': (first_at_notes-first)//4,\n            \'extrapolated_notes\': int(np.sum((notes[:, 0] < beats[0, 0]) | (notes[:, 0] > beats[-1, 0])))}\n\n\ndef detect_and_export(audio, transcription, output):\n    import collections\n    import collections.abc\n    collections.MutableSequence = collections.abc.MutableSequence\n    from madmom.features.downbeats import RNNDownBeatProcessor, DBNDownBeatTrackingProcessor\n    # RNNDownBeatProcessor reads MP3/WAV via madmom/FFmpeg, not soundfile.\n    print(\'Identificando a pulsação e o início dos compassos em 4/4…\', flush=True)\n    activations = RNNDownBeatProcessor(num_threads=1)(str(audio))\n    if len(activations) < 500 or not np.isfinite(activations).all():\n        raise ValueError(\'Áudio insuficiente para identificar os compassos.\')\n    beats = DBNDownBeatTrackingProcessor(beats_per_bar=[4], fps=100,\n                                        min_bpm=40, max_bpm=240, num_threads=1)(activations)\n    if len(beats) < 8:\n        raise ValueError(\'A análise não encontrou pulsação suficiente. Nenhum alinhamento foi aplicado.\')\n    notes = np.loadtxt(transcription, ndmin=2)\n    result = export_aligned(notes, beats, output)\n    Path(output).with_suffix(\'.ritmo.json\').write_text(json.dumps(\n        {\'source_audio\': Path(audio).name, \'beats\': beats.tolist(), **result}, indent=2))\n    print(f"Andamento mediano estimado: {result[\'median_bpm\']:.2f} BPM")\n    print(f"MIDI pronto: {result[\'notes\']} golpes · {result[\'bars\']} compassos.")\n    if result[\'leading_bars\']:\n        print(f"Introdução preservada: {result[\'leading_bars\']} compassos antes do primeiro golpe.")\n    print(\'Alinhamento incluído no MIDI. Importe-o no Compasso atualizado.\')\n    if result[\'extrapolated_notes\']:\n        print(\'Há golpes fora do trecho com pulsação detectada; a grade foi extrapolada nesses pontos.\')\n    print(\'Resultado automático estimado em 4/4; ritmos ambíguos ainda podem ser interpretados incorretamente.\')\n    return result\n\n\nif __name__ == \'__main__\':\n    import sys\n    detect_and_export(*sys.argv[1:4])\n'
MIDIAUTO = SAIDA / 'bateria-automatica.mid'
ambiente = os.environ.copy()
ambiente['OPENBLAS_NUM_THREADS'] = '2'
ambiente['OMP_NUM_THREADS'] = '2'
print('Analisando os compassos. Esta etapa pode levar alguns minutos…')
resultado = subprocess.run(
    [str(PYTHON), '-u', '-c', codigo_automatico, str(AUDIO), str(TXT), str(MIDIAUTO)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ambiente
)
print(resultado.stdout)
resultado.check_returncode()
files.download(str(MIDIAUTO))
